Extract daily/monthly data
-
From global simulation

In [1]:
cd ~/Pythons

/home/users/guicha/Pythons


In [2]:
import sys
import os
import argparse
import numpy as np

from KSCALE.read_data.read_data_catalog import *
from KSCALE.p_config import regions

/home/users/guicha/.conda/envs/hk25/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [3]:
variable = 'hfssd'
driving= 'CoMA9_TBv1p2'  # 'GAL9' # 'RAL3p3'
resolution = 'n1280'  # 'n2560'
zoom = 7
year = 2020
months = np.arange(1, 12+1, 1)

ts = '1D'  # resampling timescale
if ts == '1M':
    ts_ = 'ME'
else:
    ts_ = '1D'

conf = 'um_glm_' + resolution + '_' + driving

In [4]:
region='Africa'

lat_range = regions[region][0]
lon_range = regions[region][1]
lat_min = lat_range[0]
lat_max = lat_range[1]
lon_min = lon_range[0]
lon_max = lon_range[1]

In [5]:
#~ Outdir

if not os.path.isdir(KSCALEOUTDIR + '/' + variable):
    os.mkdir(KSCALEOUTDIR + '/' + variable)
outdir = KSCALEOUTDIR + '/' + variable

if not os.path.isdir(outdir + '/' + conf):
    os.mkdir(outdir + '/' + conf)
outdir = outdir + '/' + conf

if not os.path.isdir(outdir + '/z' + str(zoom)):
    os.mkdir(outdir + '/z' + str(zoom))
outdir = outdir + '/z' + str(zoom)

if not os.path.isdir(outdir + '/lat={0},{1}_lon={2},{3}'.format(lat_min, lat_max, lon_min, lon_max)):
    os.mkdir(outdir + '/lat={0},{1}_lon={2},{3}'.format(lat_min, lat_max, lon_min, lon_max))
outdir = outdir + '/lat={0},{1}_lon={2},{3}'.format(lat_min, lat_max, lon_min, lon_max)

outdir

'/gws/nopw/j04/kscale/USERS/guicha/outputs/data/hfssd/um_glm_n1280_CoMA9_TBv1p2/z7/lat=-35.0,25.0_lon=-20.0,52.0'

In [6]:
#~ Get data

data = get_2d_variable_global(resolution=resolution, driving=driving, zoom=zoom,
                                variable=variable, lat_range=(lat_min, lat_max), lon_range=(lon_min, lon_max))

/home/users/guicha/.conda/envs/hk25/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [7]:
iyrs = data.groupby('time.year').groups
data = data.isel(time=iyrs[year])
imths = data.groupby('time.month').groups

In [9]:
data_ms = []

for m in months:
    print(m, end=' : ', flush=True)
    data_m = data.isel(time=imths[m])

    if variable == 'pr':
        data_m = data_m.resample(time=ts_).sum() * 3600  # kg m-2 s-1 -> mm hr-1
    else:
        data_m = data_m.resample(time=ts_).mean()

    data_ms.append(data_m)

data_ms = xr.concat(data_ms, dim='time')

1 : 2 : 3 : 4 : 5 : 6 : 7 : 8 : 9 : 10 : 11 : 12 : 

In [12]:
#~ Save

outfile = outdir + '/' + str(year) + '_' + ts + '.nc'
data_ms.to_netcdf(outfile)
outfile

'/gws/nopw/j04/kscale/USERS/guicha/outputs/data/hfssd/um_glm_n1280_CoMA9_TBv1p2/z7/lat=-35.0,25.0_lon=-20.0,52.0/2020_1D.nc'